ROBUSTNESS

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_DIR = Path("/content/drive/MyDrive/diffusion_project")

print("Project exists:", PROJECT_DIR.exists())
print("ShuffleNet results:", (PROJECT_DIR / "results" / "shufflenetv2").exists())

if (PROJECT_DIR / "results" / "shufflenetv2").exists():
    print("\nFiles:")
    for p in (PROJECT_DIR / "results" / "shufflenetv2").iterdir():
        print(" -", p.name)

Project exists: True
ShuffleNet results: True

Files:
 - training_curves.png
 - results.json
 - confusion_matrix.png
 - checkpoint.pth
 - cross_dataset
 - int8_confusion_matrix.png
 - model_size_comparison.png
 - fp32_vs_int8_metrics.png
 - latency_comparison.png
 - results_int8.json


In [3]:
import torch
from pathlib import Path

CHECKPOINT = Path(
    "/content/drive/MyDrive/diffusion_project/"
    "results/shufflenetv2/checkpoint.pth"
)

checkpoint = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint loaded successfully.")
print("Type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print(" -", key)

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    else:
        state_dict = checkpoint

    print("\nNumber of parameter tensors:", len(state_dict))

    print("\nFirst 5 parameter names:")
    for i, key in enumerate(state_dict.keys()):
        if i >= 5:
            break
        print(" -", key)
else:
    print("Checkpoint is not a dictionary.")

Checkpoint loaded successfully.
Type: <class 'collections.OrderedDict'>

Checkpoint keys:
 - total_ops
 - total_params
 - conv1.0.weight
 - conv1.1.weight
 - conv1.1.bias
 - conv1.1.running_mean
 - conv1.1.running_var
 - conv1.1.num_batches_tracked
 - stage2.0.total_ops
 - stage2.0.total_params
 - stage2.0.branch1.0.weight
 - stage2.0.branch1.1.weight
 - stage2.0.branch1.1.bias
 - stage2.0.branch1.1.running_mean
 - stage2.0.branch1.1.running_var
 - stage2.0.branch1.1.num_batches_tracked
 - stage2.0.branch1.2.weight
 - stage2.0.branch1.3.weight
 - stage2.0.branch1.3.bias
 - stage2.0.branch1.3.running_mean
 - stage2.0.branch1.3.running_var
 - stage2.0.branch1.3.num_batches_tracked
 - stage2.0.branch2.0.weight
 - stage2.0.branch2.1.weight
 - stage2.0.branch2.1.bias
 - stage2.0.branch2.1.running_mean
 - stage2.0.branch2.1.running_var
 - stage2.0.branch2.1.num_batches_tracked
 - stage2.0.branch2.3.weight
 - stage2.0.branch2.4.weight
 - stage2.0.branch2.4.bias
 - stage2.0.branch2.4.running_m

In [4]:
import torch
from torchvision.models import shufflenet_v2_x1_0

CHECKPOINT = (
    "/content/drive/MyDrive/diffusion_project/"
    "results/shufflenetv2/checkpoint.pth"
)

model = shufflenet_v2_x1_0(weights=None)

# Original experiment is binary AI-vs-real classification
model.fc = torch.nn.Linear(
    model.fc.in_features,
    2
)

state_dict = torch.load(
    CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

# Remove profiling entries that are not model parameters
state_dict = {
    key: value
    for key, value in state_dict.items()
    if not key.endswith("total_ops")
    and not key.endswith("total_params")
}

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

print("Model reconstructed successfully.")
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model.eval()

dummy_input = torch.randn(1, 3, 224, 224)

with torch.no_grad():
    output = model(dummy_input)

print("Input shape:", tuple(dummy_input.shape))
print("Output shape:", tuple(output.shape))
print("Output:", output)

Model reconstructed successfully.
Missing keys: []
Unexpected keys: []
Input shape: (1, 3, 224, 224)
Output shape: (1, 2)
Output: tensor([[ 22.6627, -22.6350]])


In [5]:
from pathlib import Path

ROBUSTNESS_ROOT = Path(
    "/content/drive/MyDrive/diffusion_project/robustness_experiment"
)

# Create completely separate output directories
for folder in [
    ROBUSTNESS_ROOT / "checkpoints",
    ROBUSTNESS_ROOT / "results",
    ROBUSTNESS_ROOT / "plots",
    ROBUSTNESS_ROOT / "code",
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Robustness workspace created:")
print(ROBUSTNESS_ROOT)

for folder in [
    ROBUSTNESS_ROOT / "checkpoints",
    ROBUSTNESS_ROOT / "results",
    ROBUSTNESS_ROOT / "plots",
    ROBUSTNESS_ROOT / "code",
]:
    print(" -", folder)

Robustness workspace created:
/content/drive/MyDrive/diffusion_project/robustness_experiment
 - /content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints
 - /content/drive/MyDrive/diffusion_project/robustness_experiment/results
 - /content/drive/MyDrive/diffusion_project/robustness_experiment/plots
 - /content/drive/MyDrive/diffusion_project/robustness_experiment/code


In [7]:
from pathlib import Path
import pandas as pd

DATASET_ROOT = Path(
    "/content/drive/MyDrive/diffusion_project/dataset_final"
)

MANIFEST = DATASET_ROOT / "manifest.csv"

print("Dataset exists:", DATASET_ROOT.exists())
print("Manifest exists:", MANIFEST.exists())

df = pd.read_csv(MANIFEST)

print("\nColumns:")
print(df.columns.tolist())

print("\nTotal images:", len(df))

print("\nSplit counts:")
print(df["split"].value_counts())

print("\nClass counts:")
print(df["class"].value_counts())

print("\nSample:")
print(df.head())

Dataset exists: True
Manifest exists: True

Columns:
['class', 'split', 'original_path', 'final_path']

Total images: 5000

Split counts:
split
train    3500
val       750
test      750
Name: count, dtype: int64

Class counts:
class
ai        2500
nature    2500
Name: count, dtype: int64

Sample:
  class  split                                      original_path  \
0    ai  train  tiny_genimage_data/imagenet_ai_0424_sdv5/train...   
1    ai  train  tiny_genimage_data/imagenet_ai_0424_sdv5/train...   
2    ai  train  tiny_genimage_data/imagenet_ai_0424_sdv5/val/a...   
3    ai  train  tiny_genimage_data/imagenet_ai_0424_sdv5/val/a...   
4    ai  train  tiny_genimage_data/imagenet_ai_0424_sdv5/train...   

                                          final_path  
0  /content/drive/MyDrive/diffusion_project/datas...  
1  /content/drive/MyDrive/diffusion_project/datas...  
2  /content/drive/MyDrive/diffusion_project/datas...  
3  /content/drive/MyDrive/diffusion_project/datas...  
4  /content/

In [8]:
# Check the class-to-index mapping used by ImageFolder

from torchvision.datasets import ImageFolder
from torchvision import transforms

DATASET_ROOT = "/content/drive/MyDrive/diffusion_project/dataset_final"

transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
])

test_dataset = ImageFolder(
    root=f"{DATASET_ROOT}/test",
    transform=transform
)

print("Class names:", test_dataset.classes)
print("Class-to-index mapping:", test_dataset.class_to_idx)
print("Number of test images:", len(test_dataset))

Class names: ['ai', 'nature']
Class-to-index mapping: {'ai': 0, 'nature': 1}
Number of test images: 750


In [9]:
from pathlib import Path

ROBUSTNESS_ROOT = Path(
    "/content/drive/MyDrive/diffusion_project/robustness_experiment"
)

RESULTS_DIR = ROBUSTNESS_ROOT / "results"
PLOTS_DIR = ROBUSTNESS_ROOT / "plots"

# Safety check: these must be outside the original results directory
ORIGINAL_RESULTS = Path(
    "/content/drive/MyDrive/diffusion_project/results/shufflenetv2"
)

assert RESULTS_DIR != ORIGINAL_RESULTS
assert PLOTS_DIR != ORIGINAL_RESULTS
assert RESULTS_DIR.parent == ROBUSTNESS_ROOT
assert PLOTS_DIR.parent == ROBUSTNESS_ROOT

print("Robustness output directory:", RESULTS_DIR)
print("Plot directory:", PLOTS_DIR)
print("Original results directory:", ORIGINAL_RESULTS)
print("\nSafety check passed: original results will not be used for output.")

Robustness output directory: /content/drive/MyDrive/diffusion_project/robustness_experiment/results
Plot directory: /content/drive/MyDrive/diffusion_project/robustness_experiment/plots
Original results directory: /content/drive/MyDrive/diffusion_project/results/shufflenetv2

Safety check passed: original results will not be used for output.


In [13]:
# ============================================================
# SHUFFLENETV2 BASELINE ROBUSTNESS EVALUATION
# Clean test-set evaluation using the exact original
# preprocessing and trained checkpoint.
# ============================================================

import os
import json
import torch
import torch.nn as nn
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

# ORIGINAL — READ ONLY
ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

ORIGINAL_CHECKPOINT = os.path.join(
    ORIGINAL_RESULTS_DIR,
    "checkpoint.pth"
)

# NEW ROBUSTNESS EXPERIMENT DIRECTORY
ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

NEW_RESULTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "results"
)

os.makedirs(NEW_RESULTS_DIR, exist_ok=True)

# Output file
BASELINE_RESULT_PATH = os.path.join(
    NEW_RESULTS_DIR,
    "baseline_robustness.json"
)

# ============================================================
# 2. SAFETY CHECK
# ============================================================

assert os.path.abspath(NEW_RESULTS_DIR) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), "Safety error: output directory must not be the original results directory."

print("Original checkpoint:")
print(ORIGINAL_CHECKPOINT)

print("\nNew results directory:")
print(NEW_RESULTS_DIR)

print("\nSafety check passed.")
print("Original results will NOT be modified.")

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

# ============================================================
# 4. DATASET PATH
# ============================================================

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "test"
)

assert os.path.exists(TEST_DIR), (
    f"Test dataset not found:\n{TEST_DIR}"
)

# ============================================================
# 5. EXACT ORIGINAL EVALUATION TRANSFORM
# ============================================================
#
# This matches the original training pipeline:
#
# transforms.Resize((224, 224))
# transforms.ToTensor()
# transforms.Normalize(...)
#
# IMPORTANT:
# Do NOT change this to Resize(256) + CenterCrop(224).
# ============================================================

IMG_SIZE = 224

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ============================================================
# 6. LOAD TEST DATASET
# ============================================================

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transform
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print("\nTest images:", len(test_dataset))
print("Classes:", test_dataset.class_to_idx)

# ============================================================
# 7. RECONSTRUCT SHUFFLENETV2
# ============================================================

try:
    import timm

    try:
        model = timm.create_model(
            "shufflenetv2_x1_0",
            pretrained=True,
            num_classes=2
        )

        print("\nUsing timm ShuffleNetV2.")

    except Exception:
        from torchvision.models import shufflenet_v2_x1_0

        model = shufflenet_v2_x1_0(
            weights="DEFAULT"
        )

        model.fc = nn.Linear(
            model.fc.in_features,
            2
        )

        print("\nUsing torchvision ShuffleNetV2.")

except Exception:
    from torchvision.models import shufflenet_v2_x1_0

    model = shufflenet_v2_x1_0(
        weights="DEFAULT"
    )

    model.fc = nn.Linear(
        model.fc.in_features,
        2
    )

    print("\nUsing torchvision ShuffleNetV2.")

# ============================================================
# 8. LOAD ORIGINAL TRAINED CHECKPOINT
# ============================================================

print("\nLoading original trained checkpoint...")

checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        f"Unexpected checkpoint type: {type(checkpoint)}"
    )

# The original checkpoint contains both:
#   1. actual model parameters
#   2. profiling metadata such as total_ops and total_params
#
# Remove only the profiling metadata before loading.
# The original checkpoint itself is NOT modified.

state_dict = {
    key: value
    for key, value in checkpoint.items()
    if not (
        key == "total_ops"
        or key == "total_params"
        or key.endswith(".total_ops")
        or key.endswith(".total_params")
    )
}

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

assert len(missing_keys) == 0, (
    f"Missing model parameters: {missing_keys}"
)

assert len(unexpected_keys) == 0, (
    f"Unexpected model parameters: {unexpected_keys}"
)

model = model.to(device)
model.eval()

print("Model loaded successfully.")
# ============================================================
# 9. CLEAN BASELINE EVALUATION
# ============================================================

print("\nRunning clean baseline evaluation...")

all_labels = []
all_predictions = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        outputs = model(images)

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.cpu().numpy().tolist()
        )

        all_predictions.extend(
            predictions.cpu().numpy().tolist()
        )

# ============================================================
# 10. CALCULATE METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)

baseline_results = {
    "model": "ShuffleNetV2",
    "evaluation_type": "clean_baseline",
    "dataset": "AI-vs-Real",
    "num_images": len(test_dataset),
    "input_size": "224x224",
    "preprocessing": {
        "resize": "224x224",
        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],
        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },
    "accuracy": float(accuracy),
    "precision": float(precision),
    "recall": float(recall),
    "f1": float(f1)
}

# ============================================================
# 11. PRINT RESULTS
# ============================================================

print("\n" + "=" * 50)
print("CLEAN BASELINE RESULTS")
print("=" * 50)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ============================================================
# 12. SAVE RESULTS
# ============================================================

with open(
    BASELINE_RESULT_PATH,
    "w"
) as f:

    json.dump(
        baseline_results,
        f,
        indent=4
    )

print("\nSaved baseline result to:")
print(BASELINE_RESULT_PATH)

print("\nBaseline evaluation complete.")

Original checkpoint:
/content/drive/MyDrive/diffusion_project/results/shufflenetv2/checkpoint.pth

New results directory:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results

Safety check passed.
Original results will NOT be modified.

Device: cuda

Test images: 750
Classes: {'ai': 0, 'nature': 1}

Using torchvision ShuffleNetV2.

Loading original trained checkpoint...
Missing keys: []
Unexpected keys: []
Model loaded successfully.

Running clean baseline evaluation...

CLEAN BASELINE RESULTS
Accuracy : 0.9853
Precision: 0.9789
Recall   : 0.9920
F1 Score : 0.9854

Saved baseline result to:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results/baseline_robustness.json

Baseline evaluation complete.


In [14]:
# ============================================================
# SHUFFLENETV2 BASELINE CORRUPTION EVALUATION
# ============================================================
#
# Evaluates the ORIGINAL trained ShuffleNetV2 checkpoint on:
#   1. Clean
#   2. JPEG compression
#   3. Gaussian blur
#   4. Gaussian noise
#   5. Low-resolution
#
# IMPORTANT:
# - Original checkpoint is READ ONLY.
# - Original results directory is READ ONLY.
# - All new outputs are saved under robustness_experiment/.
# ============================================================

import os
import json
import io
import random
import numpy as np

import torch
import torch.nn as nn

from PIL import Image, ImageFilter
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

# ORIGINAL — READ ONLY
ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

ORIGINAL_CHECKPOINT = os.path.join(
    ORIGINAL_RESULTS_DIR,
    "checkpoint.pth"
)

# NEW ROBUSTNESS EXPERIMENT
ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

NEW_RESULTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "results"
)

NEW_PLOTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "plots"
)

os.makedirs(NEW_RESULTS_DIR, exist_ok=True)
os.makedirs(NEW_PLOTS_DIR, exist_ok=True)

OUTPUT_PATH = os.path.join(
    NEW_RESULTS_DIR,
    "baseline_corruptions.json"
)

# ============================================================
# 2. SAFETY CHECKS
# ============================================================

assert os.path.abspath(NEW_RESULTS_DIR) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), "Safety error: output directory cannot be original results directory."

assert os.path.exists(ORIGINAL_CHECKPOINT), (
    f"Original checkpoint not found:\n{ORIGINAL_CHECKPOINT}"
)

print("Original checkpoint:")
print(ORIGINAL_CHECKPOINT)

print("\nNew results directory:")
print(NEW_RESULTS_DIR)

print("\nSafety check passed.")
print("Original results will NOT be modified.")

# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)

# ============================================================
# 4. DATASET
# ============================================================

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "test"
)

assert os.path.exists(TEST_DIR), (
    f"Test dataset not found:\n{TEST_DIR}"
)

# ============================================================
# 5. NORMALIZATION
# ============================================================

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# ============================================================
# 6. CORRUPTION TRANSFORMS
# ============================================================

class JPEGCompression:
    """
    Simulates JPEG compression artifacts.
    """

    def __init__(self, quality=30):
        self.quality = quality

    def __call__(self, image):
        buffer = io.BytesIO()

        image.save(
            buffer,
            format="JPEG",
            quality=self.quality
        )

        buffer.seek(0)

        compressed = Image.open(buffer).convert("RGB")

        return compressed


class GaussianNoise:
    """
    Adds Gaussian noise to the image.
    """

    def __init__(self, std=0.08):
        self.std = std

    def __call__(self, image):

        image_np = np.asarray(
            image
        ).astype(np.float32) / 255.0

        noise = np.random.normal(
            loc=0.0,
            scale=self.std,
            size=image_np.shape
        )

        noisy = image_np + noise

        noisy = np.clip(
            noisy,
            0.0,
            1.0
        )

        noisy = (
            noisy * 255.0
        ).astype(np.uint8)

        return Image.fromarray(
            noisy
        )


class LowResolution:
    """
    Downsamples the image and then restores it to 224x224.
    """

    def __init__(self, low_size=56):
        self.low_size = low_size

    def __call__(self, image):

        image = image.resize(
            (self.low_size, self.low_size),
            Image.Resampling.BILINEAR
        )

        image = image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        return image


class FixedGaussianBlur:
    """
    Applies Gaussian blur.
    """

    def __init__(self, radius=2.0):
        self.radius = radius

    def __call__(self, image):

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=self.radius
            )
        )


# ============================================================
# 7. EXACT ORIGINAL PREPROCESSING
# ============================================================
#
# Original pipeline used:
#
# Resize((224,224))
# ToTensor()
# Normalize(...)
#
# For corruptions, the corruption is applied first,
# followed by the exact original resize + normalization.
# ============================================================

def make_transform(corruption=None):

    transform_list = []

    if corruption is not None:
        transform_list.append(
            corruption
        )

    transform_list.extend([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        NORMALIZE
    ])

    return transforms.Compose(
        transform_list
    )


# ============================================================
# 8. LOAD SHUFFLENETV2
# ============================================================

from torchvision.models import shufflenet_v2_x1_0

model = shufflenet_v2_x1_0(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

# ============================================================
# 9. LOAD ORIGINAL CHECKPOINT
# ============================================================

print("\nLoading original trained checkpoint...")

checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

if not isinstance(checkpoint, dict):
    raise TypeError(
        f"Unexpected checkpoint type: {type(checkpoint)}"
    )

# Remove profiling metadata only.
# The original checkpoint itself is NOT modified.

state_dict = {
    key: value
    for key, value in checkpoint.items()
    if not (
        key == "total_ops"
        or key == "total_params"
        or key.endswith(".total_ops")
        or key.endswith(".total_params")
    )
}

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False
)

print("Missing keys:", missing_keys)
print("Unexpected keys:", unexpected_keys)

assert len(missing_keys) == 0, (
    f"Missing model parameters: {missing_keys}"
)

assert len(unexpected_keys) == 0, (
    f"Unexpected model parameters: {unexpected_keys}"
)

model = model.to(device)
model.eval()

print("Model loaded successfully.")

# ============================================================
# 10. EVALUATION FUNCTION
# ============================================================

def evaluate_dataset(dataset):

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model(images)

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "num_images": len(dataset)
    }


# ============================================================
# 11. DEFINE CORRUPTIONS
# ============================================================

corruptions = {
    "clean": None,

    "jpeg": JPEGCompression(
        quality=30
    ),

    "gaussian_blur": FixedGaussianBlur(
        radius=2.0
    ),

    "gaussian_noise": GaussianNoise(
        std=0.08
    ),

    "low_resolution": LowResolution(
        low_size=56
    )
}

# ============================================================
# 12. RUN EVALUATIONS
# ============================================================

results = {
    "model": "ShuffleNetV2",
    "evaluation_type": "baseline_corruption_evaluation",
    "dataset": "AI-vs-Real",
    "input_size": "224x224",
    "preprocessing": {
        "resize": "224x224",
        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],
        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },
    "corruptions": {}
}

print("\n" + "=" * 60)
print("BASELINE CORRUPTION EVALUATION")
print("=" * 60)

for corruption_name, corruption in corruptions.items():

    print(
        f"\nEvaluating: {corruption_name}"
    )

    dataset = datasets.ImageFolder(
        TEST_DIR,
        transform=make_transform(
            corruption
        )
    )

    metrics = evaluate_dataset(
        dataset
    )

    results["corruptions"][
        corruption_name
    ] = metrics

    print(
        f"Accuracy : {metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: {metrics['precision']:.4f}"
    )

    print(
        f"Recall   : {metrics['recall']:.4f}"
    )

    print(
        f"F1 Score : {metrics['f1']:.4f}"
    )

# ============================================================
# 13. SAVE RESULTS
# ============================================================

with open(
    OUTPUT_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )

print("\n" + "=" * 60)
print("BASELINE CORRUPTION EVALUATION COMPLETE")
print("=" * 60)

print("\nSaved results to:")
print(OUTPUT_PATH)

print("\nOriginal results directory was not modified.")

Original checkpoint:
/content/drive/MyDrive/diffusion_project/results/shufflenetv2/checkpoint.pth

New results directory:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results

Safety check passed.
Original results will NOT be modified.

Device: cuda

Loading original trained checkpoint...
Missing keys: []
Unexpected keys: []
Model loaded successfully.

BASELINE CORRUPTION EVALUATION

Evaluating: clean
Accuracy : 0.9853
Precision: 0.9789
Recall   : 0.9920
F1 Score : 0.9854

Evaluating: jpeg
Accuracy : 0.5133
Precision: 0.5068
Recall   : 1.0000
F1 Score : 0.6726

Evaluating: gaussian_blur
Accuracy : 0.6053
Precision: 0.5590
Recall   : 0.9973
F1 Score : 0.7165

Evaluating: gaussian_noise
Accuracy : 0.5253
Precision: 0.9524
Recall   : 0.0533
F1 Score : 0.1010

Evaluating: low_resolution
Accuracy : 0.5413
Precision: 0.5216
Recall   : 1.0000
F1 Score : 0.6856

BASELINE CORRUPTION EVALUATION COMPLETE

Saved results to:
/content/drive/MyDrive/diffusion_project/robustness_expe

In [15]:
# ============================================================
# SHUFFLENETV2 ROBUSTNESS TRAINING
# ============================================================
#
# Starts from the ORIGINAL trained ShuffleNetV2 checkpoint
# and performs corruption-augmented fine-tuning.
#
# ORIGINAL CHECKPOINT:
#   READ ONLY
#
# NEW CHECKPOINT:
#   robustness_experiment/checkpoints/
#
# ============================================================

import os
import io
import random
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

from PIL import Image, ImageFilter
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torchvision.models import shufflenet_v2_x1_0


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

# ORIGINAL — READ ONLY
ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

ORIGINAL_CHECKPOINT = os.path.join(
    ORIGINAL_RESULTS_DIR,
    "checkpoint.pth"
)

# NEW ROBUSTNESS EXPERIMENT
ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

CHECKPOINT_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "checkpoints"
)

os.makedirs(
    CHECKPOINT_DIR,
    exist_ok=True
)

ROBUST_CHECKPOINT = os.path.join(
    CHECKPOINT_DIR,
    "shufflenetv2_robust_fp32.pth"
)


# ============================================================
# 2. SAFETY CHECKS
# ============================================================

assert os.path.exists(
    ORIGINAL_CHECKPOINT
), "Original checkpoint not found."

assert os.path.abspath(
    CHECKPOINT_DIR
) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), "Safety error: output directory is original results directory."

print("Original checkpoint:")
print(ORIGINAL_CHECKPOINT)

print("\nNew checkpoint:")
print(ROBUST_CHECKPOINT)

print("\nSafety check passed.")
print("Original results will NOT be modified.")


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 4. DATASET PATHS
# ============================================================

TRAIN_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "train"
)

VAL_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "val"
)

assert os.path.exists(
    TRAIN_DIR
), "Training dataset not found."

assert os.path.exists(
    VAL_DIR
), "Validation dataset not found."


# ============================================================
# 5. CORRUPTION TRANSFORMS
# ============================================================

class JPEGCompression:

    def __init__(self, quality=30):
        self.quality = quality

    def __call__(self, image):

        buffer = io.BytesIO()

        image.save(
            buffer,
            format="JPEG",
            quality=self.quality
        )

        buffer.seek(0)

        return Image.open(
            buffer
        ).convert("RGB")


class GaussianNoise:

    def __init__(self, std=0.08):
        self.std = std

    def __call__(self, image):

        image_np = np.asarray(
            image
        ).astype(
            np.float32
        ) / 255.0

        noise = np.random.normal(
            0.0,
            self.std,
            image_np.shape
        )

        noisy = image_np + noise

        noisy = np.clip(
            noisy,
            0.0,
            1.0
        )

        noisy = (
            noisy * 255.0
        ).astype(
            np.uint8
        )

        return Image.fromarray(
            noisy
        )


class LowResolution:

    def __init__(self, low_size=56):
        self.low_size = low_size

    def __call__(self, image):

        image = image.resize(
            (self.low_size, self.low_size),
            Image.Resampling.BILINEAR
        )

        image = image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        return image


class GaussianBlur:

    def __init__(self, radius=2.0):
        self.radius = radius

    def __call__(self, image):

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=self.radius
            )
        )


# ============================================================
# 6. IMAGE TRANSFORMS
# ============================================================

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


# Base spatial augmentation
BASE_AUGMENTATION = transforms.Compose([
    transforms.RandomResizedCrop(
        224
    ),
    transforms.RandomHorizontalFlip()
])


# Random corruption selection
CORRUPTION_TRANSFORMS = [
    JPEGCompression(quality=30),
    GaussianBlur(radius=2.0),
    GaussianNoise(std=0.08),
    LowResolution(low_size=56)
]


class RobustTrainingTransform:

    def __init__(
        self,
        corruption_probability=0.8
    ):
        self.corruption_probability = (
            corruption_probability
        )

    def __call__(self, image):

        # Original spatial augmentation
        image = BASE_AUGMENTATION(
            image
        )

        # Apply corruption with probability 0.8.
        # Otherwise keep the image clean.
        if random.random() < self.corruption_probability:

            corruption = random.choice(
                CORRUPTION_TRANSFORMS
            )

            image = corruption(
                image
            )

        # Match original training preprocessing
        image = transforms.ToTensor()(
            image
        )

        image = NORMALIZE(
            image
        )

        return image


# Validation remains CLEAN.
# This lets us measure whether robustness training
# damages normal clean performance.

VAL_TRANSFORM = transforms.Compose([
    transforms.Resize(
        (224, 224)
    ),
    transforms.ToTensor(),
    NORMALIZE
])


# ============================================================
# 7. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=RobustTrainingTransform(
        corruption_probability=0.8
    )
)

val_dataset = datasets.ImageFolder(
    VAL_DIR,
    transform=VAL_TRANSFORM
)

print("\nTraining images:", len(train_dataset))
print("Validation images:", len(val_dataset))

print(
    "Classes:",
    train_dataset.class_to_idx
)


# ============================================================
# 8. DATA LOADERS
# ============================================================

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)


# ============================================================
# 9. RECONSTRUCT SHUFFLENETV2
# ============================================================

model = shufflenet_v2_x1_0(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)


# ============================================================
# 10. LOAD ORIGINAL TRAINED CHECKPOINT
# ============================================================

print(
    "\nLoading original trained checkpoint..."
)

checkpoint = torch.load(
    ORIGINAL_CHECKPOINT,
    map_location="cpu",
    weights_only=False
)

if not isinstance(
    checkpoint,
    dict
):
    raise TypeError(
        f"Unexpected checkpoint type: {type(checkpoint)}"
    )


# Remove profiling metadata.
# The original file is NOT changed.

state_dict = {
    key: value
    for key, value in checkpoint.items()
    if not (
        key == "total_ops"
        or key == "total_params"
        or key.endswith(".total_ops")
        or key.endswith(".total_params")
    )
}


missing_keys, unexpected_keys = (
    model.load_state_dict(
        state_dict,
        strict=False
    )
)

print(
    "Missing keys:",
    missing_keys
)

print(
    "Unexpected keys:",
    unexpected_keys
)

assert len(
    missing_keys
) == 0

assert len(
    unexpected_keys
) == 0

model = model.to(
    device
)

print(
    "Original trained ShuffleNetV2 loaded."
)


# ============================================================
# 11. TRAINING CONFIGURATION
# ============================================================

EPOCHS = 15

LEARNING_RATE = 1e-5

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)


# ============================================================
# 12. VALIDATION FUNCTION
# ============================================================

def evaluate_clean():

    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)

    return correct / total


# ============================================================
# 13. ROBUSTNESS TRAINING
# ============================================================

print("\n" + "=" * 60)
print("ROBUSTNESS TRAINING")
print("=" * 60)

best_val_accuracy = 0.0

for epoch in range(
    1,
    EPOCHS + 1
):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        optimizer.zero_grad()

        outputs = model(
            images
        )

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += (
            loss.item()
            * labels.size(0)
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

    train_loss = (
        running_loss / total
    )

    train_accuracy = (
        correct / total
    )

    val_accuracy = evaluate_clean()

    print(
        f"Epoch {epoch:02d}/{EPOCHS} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Clean Val Acc: {val_accuracy:.4f}"
    )

    # Save the best robustness-trained model
    if val_accuracy > best_val_accuracy:

        best_val_accuracy = val_accuracy

        torch.save(
            model.state_dict(),
            ROBUST_CHECKPOINT
        )

        print(
            f"  Best model saved: "
            f"{best_val_accuracy:.4f}"
        )


# ============================================================
# 14. VERIFY OUTPUT
# ============================================================

assert os.path.exists(
    ROBUST_CHECKPOINT
), "Robust checkpoint was not created."

print("\n" + "=" * 60)
print("ROBUSTNESS TRAINING COMPLETE")
print("=" * 60)

print(
    "\nBest clean validation accuracy:",
    f"{best_val_accuracy:.4f}"
)

print(
    "\nRobust FP32 checkpoint saved to:"
)

print(
    ROBUST_CHECKPOINT
)

print(
    "\nOriginal checkpoint was not modified."
)

Original checkpoint:
/content/drive/MyDrive/diffusion_project/results/shufflenetv2/checkpoint.pth

New checkpoint:
/content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints/shufflenetv2_robust_fp32.pth

Safety check passed.
Original results will NOT be modified.

Device: cuda

Training images: 3500
Validation images: 750
Classes: {'ai': 0, 'nature': 1}

Loading original trained checkpoint...
Missing keys: []
Unexpected keys: []
Original trained ShuffleNetV2 loaded.

ROBUSTNESS TRAINING
Epoch 01/15 | Train Loss: 1.5729 | Train Acc: 0.7763 | Clean Val Acc: 0.8120
  Best model saved: 0.8120
Epoch 02/15 | Train Loss: 1.0582 | Train Acc: 0.8126 | Clean Val Acc: 0.8440
  Best model saved: 0.8440
Epoch 03/15 | Train Loss: 0.7848 | Train Acc: 0.8366 | Clean Val Acc: 0.8627
  Best model saved: 0.8627
Epoch 04/15 | Train Loss: 0.6437 | Train Acc: 0.8457 | Clean Val Acc: 0.8867
  Best model saved: 0.8867
Epoch 05/15 | Train Loss: 0.5615 | Train Acc: 0.8586 | Clean Val Acc: 0.8893

In [16]:
# ============================================================
# SHUFFLENETV2 ROBUST FP32 EVALUATION
# ============================================================
#
# Evaluates the robustness-trained ShuffleNetV2 FP32 model on:
#   1. Clean
#   2. JPEG compression
#   3. Gaussian blur
#   4. Gaussian noise
#   5. Low resolution
#
# Uses the EXACT SAME corruption settings as the baseline
# corruption evaluation.
#
# Original checkpoint/results are READ ONLY.
# ============================================================

import os
import io
import json
import numpy as np

import torch
import torch.nn as nn

from PIL import Image, ImageFilter
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
from torchvision.models import shufflenet_v2_x1_0

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

# ORIGINAL — READ ONLY
ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

# ROBUST FP32 CHECKPOINT
ROBUST_CHECKPOINT = os.path.join(
    PROJECT_DIR,
    "robustness_experiment",
    "checkpoints",
    "shufflenetv2_robust_fp32.pth"
)

# NEW RESULTS DIRECTORY
NEW_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment",
    "results"
)

os.makedirs(
    NEW_RESULTS_DIR,
    exist_ok=True
)

OUTPUT_PATH = os.path.join(
    NEW_RESULTS_DIR,
    "robust_fp32.json"
)


# ============================================================
# 2. SAFETY CHECKS
# ============================================================

assert os.path.exists(
    ROBUST_CHECKPOINT
), (
    f"Robust checkpoint not found:\n"
    f"{ROBUST_CHECKPOINT}"
)

assert os.path.abspath(
    NEW_RESULTS_DIR
) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), (
    "Safety error: output directory cannot be "
    "the original results directory."
)

print("Robust FP32 checkpoint:")
print(ROBUST_CHECKPOINT)

print("\nNew results directory:")
print(NEW_RESULTS_DIR)

print("\nSafety check passed.")
print("Original results will NOT be modified.")


# ============================================================
# 3. DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("\nDevice:", device)


# ============================================================
# 4. DATASET
# ============================================================

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "test"
)

assert os.path.exists(
    TEST_DIR
), (
    f"Test dataset not found:\n{TEST_DIR}"
)


# ============================================================
# 5. NORMALIZATION
# ============================================================

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


# ============================================================
# 6. SAME CORRUPTION CLASSES USED IN BASELINE
# ============================================================

class JPEGCompression:

    def __init__(self, quality=30):
        self.quality = quality

    def __call__(self, image):

        buffer = io.BytesIO()

        image.save(
            buffer,
            format="JPEG",
            quality=self.quality
        )

        buffer.seek(0)

        return Image.open(
            buffer
        ).convert("RGB")


class GaussianNoise:

    def __init__(self, std=0.08):
        self.std = std

    def __call__(self, image):

        image_np = np.asarray(
            image
        ).astype(
            np.float32
        ) / 255.0

        noise = np.random.normal(
            0.0,
            self.std,
            image_np.shape
        )

        noisy = image_np + noise

        noisy = np.clip(
            noisy,
            0.0,
            1.0
        )

        noisy = (
            noisy * 255.0
        ).astype(
            np.uint8
        )

        return Image.fromarray(
            noisy
        )


class LowResolution:

    def __init__(self, low_size=56):
        self.low_size = low_size

    def __call__(self, image):

        image = image.resize(
            (self.low_size, self.low_size),
            Image.Resampling.BILINEAR
        )

        image = image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        return image


class FixedGaussianBlur:

    def __init__(self, radius=2.0):
        self.radius = radius

    def __call__(self, image):

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=self.radius
            )
        )


# ============================================================
# 7. TRANSFORM BUILDER
# ============================================================

def make_transform(corruption=None):

    transform_list = []

    if corruption is not None:

        transform_list.append(
            corruption
        )

    transform_list.extend([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        NORMALIZE
    ])

    return transforms.Compose(
        transform_list
    )


# ============================================================
# 8. LOAD ROBUST SHUFFLENETV2
# ============================================================

model = shufflenet_v2_x1_0(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)


# ============================================================
# 9. LOAD ROBUST FP32 CHECKPOINT
# ============================================================

print(
    "\nLoading robust FP32 checkpoint..."
)

state_dict = torch.load(
    ROBUST_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

missing_keys, unexpected_keys = (
    model.load_state_dict(
        state_dict,
        strict=True
    )
)

print(
    "Missing keys:",
    missing_keys
)

print(
    "Unexpected keys:",
    unexpected_keys
)

model = model.to(
    device
)

model.eval()

print(
    "Robust FP32 model loaded successfully."
)


# ============================================================
# 10. EVALUATION FUNCTION
# ============================================================

def evaluate_dataset(dataset):

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2,
        pin_memory=True
    )

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            images = images.to(
                device,
                non_blocking=True
            )

            outputs = model(
                images
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_labels.extend(
                labels.cpu().numpy().tolist()
            )

            all_predictions.extend(
                predictions.cpu().numpy().tolist()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "num_images": len(dataset)
    }


# ============================================================
# 11. SAME CORRUPTIONS AS BASELINE
# ============================================================

corruptions = {

    "clean": None,

    "jpeg": JPEGCompression(
        quality=30
    ),

    "gaussian_blur": FixedGaussianBlur(
        radius=2.0
    ),

    "gaussian_noise": GaussianNoise(
        std=0.08
    ),

    "low_resolution": LowResolution(
        low_size=56
    )
}


# ============================================================
# 12. RUN EVALUATIONS
# ============================================================

results = {

    "model": "ShuffleNetV2",

    "evaluation_type":
        "robust_fp32_corruption_evaluation",

    "dataset":
        "AI-vs-Real",

    "checkpoint":
        ROBUST_CHECKPOINT,

    "input_size":
        "224x224",

    "preprocessing": {

        "resize":
            "224x224",

        "normalization_mean": [
            0.485,
            0.456,
            0.406
        ],

        "normalization_std": [
            0.229,
            0.224,
            0.225
        ]
    },

    "corruptions": {}
}


print("\n" + "=" * 60)
print("ROBUST FP32 CORRUPTION EVALUATION")
print("=" * 60)


for corruption_name, corruption in corruptions.items():

    print(
        f"\nEvaluating: {corruption_name}"
    )

    dataset = datasets.ImageFolder(
        TEST_DIR,
        transform=make_transform(
            corruption
        )
    )

    metrics = evaluate_dataset(
        dataset
    )

    results[
        "corruptions"
    ][
        corruption_name
    ] = metrics

    print(
        f"Accuracy : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Recall   : "
        f"{metrics['recall']:.4f}"
    )

    print(
        f"F1 Score : "
        f"{metrics['f1']:.4f}"
    )


# ============================================================
# 13. SAVE RESULTS
# ============================================================

with open(
    OUTPUT_PATH,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


# ============================================================
# 14. COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("ROBUST FP32 EVALUATION COMPLETE")
print("=" * 60)

print("\nSaved results to:")
print(OUTPUT_PATH)

print("\nOriginal results directory was not modified.")

Robust FP32 checkpoint:
/content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints/shufflenetv2_robust_fp32.pth

New results directory:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results

Safety check passed.
Original results will NOT be modified.

Device: cuda

Loading robust FP32 checkpoint...
Missing keys: []
Unexpected keys: []
Robust FP32 model loaded successfully.

ROBUST FP32 CORRUPTION EVALUATION

Evaluating: clean
Accuracy : 0.9280
Precision: 0.9791
Recall   : 0.8747
F1 Score : 0.9239

Evaluating: jpeg
Accuracy : 0.8547
Precision: 0.8050
Recall   : 0.9360
F1 Score : 0.8656

Evaluating: gaussian_blur
Accuracy : 0.9160
Precision: 0.9239
Recall   : 0.9067
F1 Score : 0.9152

Evaluating: gaussian_noise
Accuracy : 0.8733
Precision: 0.8500
Recall   : 0.9067
F1 Score : 0.8774

Evaluating: low_resolution
Accuracy : 0.8773
Precision: 0.8410
Recall   : 0.9307
F1 Score : 0.8835

ROBUST FP32 EVALUATION COMPLETE

Saved results to:
/content/drive/MyDrive/d

In [17]:
# ============================================================
# ORIGINAL vs ROBUST SHUFFLENETV2 COMPARISON
# ============================================================

import os
import json
import matplotlib.pyplot as plt


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

RESULTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "results"
)

PLOTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "plots"
)

ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

BASELINE_FILE = os.path.join(
    RESULTS_DIR,
    "baseline_corruptions.json"
)

ROBUST_FILE = os.path.join(
    RESULTS_DIR,
    "robust_fp32.json"
)

COMPARISON_FILE = os.path.join(
    RESULTS_DIR,
    "baseline_vs_robust_fp32.json"
)

PLOT_FILE = os.path.join(
    PLOTS_DIR,
    "baseline_vs_robust_fp32.png"
)

os.makedirs(
    PLOTS_DIR,
    exist_ok=True
)


# ============================================================
# 2. SAFETY CHECK
# ============================================================

assert os.path.abspath(
    RESULTS_DIR
) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), (
    "Safety error: original results directory "
    "cannot be used for output."
)

assert os.path.exists(
    BASELINE_FILE
), (
    f"Baseline results not found:\n{BASELINE_FILE}"
)

assert os.path.exists(
    ROBUST_FILE
), (
    f"Robust results not found:\n{ROBUST_FILE}"
)

print("Safety check passed.")
print("Original results will NOT be modified.")


# ============================================================
# 3. LOAD RESULTS
# ============================================================

with open(BASELINE_FILE, "r") as f:
    baseline = json.load(f)

with open(ROBUST_FILE, "r") as f:
    robust = json.load(f)


# ============================================================
# 4. CONDITIONS
# ============================================================

conditions = [
    "clean",
    "jpeg",
    "gaussian_blur",
    "gaussian_noise",
    "low_resolution"
]


# ============================================================
# 5. BUILD COMPARISON
# ============================================================

comparison = {
    "model": "ShuffleNetV2",
    "comparison": "Original FP32 vs Robust FP32",
    "dataset": "AI-vs-Real",
    "conditions": {}
}

for condition in conditions:

    baseline_metrics = baseline[
        "corruptions"
    ][condition]

    robust_metrics = robust[
        "corruptions"
    ][condition]

    comparison["conditions"][condition] = {

        "baseline": baseline_metrics,

        "robust": robust_metrics,

        "accuracy_change": (
            robust_metrics["accuracy"]
            - baseline_metrics["accuracy"]
        ),

        "f1_change": (
            robust_metrics["f1"]
            - baseline_metrics["f1"]
        )
    }


# ============================================================
# 6. SAVE COMPARISON JSON
# ============================================================

with open(
    COMPARISON_FILE,
    "w"
) as f:

    json.dump(
        comparison,
        f,
        indent=4
    )


# ============================================================
# 7. PRINT SUMMARY
# ============================================================

print("\n" + "=" * 70)
print("ORIGINAL vs ROBUST FP32")
print("=" * 70)

for condition in conditions:

    baseline_acc = comparison[
        "conditions"
    ][condition]["baseline"]["accuracy"]

    robust_acc = comparison[
        "conditions"
    ][condition]["robust"]["accuracy"]

    change = comparison[
        "conditions"
    ][condition]["accuracy_change"]

    baseline_f1 = comparison[
        "conditions"
    ][condition]["baseline"]["f1"]

    robust_f1 = comparison[
        "conditions"
    ][condition]["robust"]["f1"]

    f1_change = comparison[
        "conditions"
    ][condition]["f1_change"]

    print(f"\n{condition}")

    print(
        f"Accuracy: "
        f"{baseline_acc:.4f} -> "
        f"{robust_acc:.4f} "
        f"({change:+.4f})"
    )

    print(
        f"F1      : "
        f"{baseline_f1:.4f} -> "
        f"{robust_f1:.4f} "
        f"({f1_change:+.4f})"
    )


# ============================================================
# 8. PREPARE PLOT DATA
# ============================================================

labels = [
    "Clean",
    "JPEG",
    "Gaussian\nBlur",
    "Gaussian\nNoise",
    "Low\nResolution"
]

baseline_accuracy = [
    comparison["conditions"][c]["baseline"]["accuracy"] * 100
    for c in conditions
]

robust_accuracy = [
    comparison["conditions"][c]["robust"]["accuracy"] * 100
    for c in conditions
]

baseline_f1 = [
    comparison["conditions"][c]["baseline"]["f1"] * 100
    for c in conditions
]

robust_f1 = [
    comparison["conditions"][c]["robust"]["f1"] * 100
    for c in conditions
]


# ============================================================
# 9. ACCURACY COMPARISON PLOT
# ============================================================

x = range(len(labels))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(
    [i - width / 2 for i in x],
    baseline_accuracy,
    width,
    label="Original FP32"
)

plt.bar(
    [i + width / 2 for i in x],
    robust_accuracy,
    width,
    label="Robust FP32"
)

plt.xticks(
    list(x),
    labels
)

plt.ylabel("Accuracy (%)")
plt.xlabel("Evaluation Condition")

plt.title(
    "ShuffleNetV2: Original vs Robust FP32 Accuracy"
)

plt.ylim(
    0,
    100
)

plt.legend()

plt.tight_layout()

plt.savefig(
    PLOT_FILE,
    dpi=300,
    bbox_inches="tight"
)

plt.close()


# ============================================================
# 10. COMPLETE
# ============================================================

print("\n" + "=" * 70)
print("COMPARISON COMPLETE")
print("=" * 70)

print("\nComparison JSON:")
print(COMPARISON_FILE)

print("\nComparison plot:")
print(PLOT_FILE)

print("\nOriginal results directory was not modified.")

Safety check passed.
Original results will NOT be modified.

ORIGINAL vs ROBUST FP32

clean
Accuracy: 0.9853 -> 0.9280 (-0.0573)
F1      : 0.9854 -> 0.9239 (-0.0615)

jpeg
Accuracy: 0.5133 -> 0.8547 (+0.3413)
F1      : 0.6726 -> 0.8656 (+0.1930)

gaussian_blur
Accuracy: 0.6053 -> 0.9160 (+0.3107)
F1      : 0.7165 -> 0.9152 (+0.1987)

gaussian_noise
Accuracy: 0.5253 -> 0.8733 (+0.3480)
F1      : 0.1010 -> 0.8774 (+0.7764)

low_resolution
Accuracy: 0.5413 -> 0.8773 (+0.3360)
F1      : 0.6856 -> 0.8835 (+0.1980)

COMPARISON COMPLETE

Comparison JSON:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results/baseline_vs_robust_fp32.json

Comparison plot:
/content/drive/MyDrive/diffusion_project/robustness_experiment/plots/baseline_vs_robust_fp32.png

Original results directory was not modified.


In [19]:
import torch

print("PyTorch version:", torch.__version__)
print("Quantized engine:", torch.backends.quantized.engine)
print("Supported engines:", torch.backends.quantized.supported_engines)

print("\nCPU quantized conv2d available:")
try:
    x = torch.quantize_per_tensor(
        torch.randn(1, 3, 224, 224),
        scale=0.1,
        zero_point=0,
        dtype=torch.quint8
    )

    w = torch.randn(8, 3, 3, 3)

    qweight = torch.quantize_per_tensor(
        w,
        scale=0.1,
        zero_point=0,
        dtype=torch.qint8
    )

    print("Quantized tensor test created successfully.")
except Exception as e:
    print("Quantized tensor test failed:")
    print(type(e).__name__, str(e))

PyTorch version: 2.11.0+cu128
Quantized engine: fbgemm
Supported engines: ['qnnpack', 'onednn', 'x86', 'fbgemm']

CPU quantized conv2d available:
Quantized tensor test created successfully.


In [20]:
# ============================================================
# ROBUST SHUFFLENETV2 -> INT8
# FX GRAPH-MODE STATIC QUANTIZATION
# ============================================================

import os
import json

import torch
import torch.nn as nn

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import shufflenet_v2_x1_0

from torch.ao.quantization import get_default_qconfig
from torch.ao.quantization.quantize_fx import (
    prepare_fx,
    convert_fx
)

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

FP32_CHECKPOINT = os.path.join(
    ROBUSTNESS_DIR,
    "checkpoints",
    "shufflenetv2_robust_fp32.pth"
)

INT8_CHECKPOINT = os.path.join(
    ROBUSTNESS_DIR,
    "checkpoints",
    "shufflenetv2_robust_int8.pth"
)

TRAIN_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "train"
)

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "test"
)

RESULTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "results"
)

OUTPUT_FILE = os.path.join(
    RESULTS_DIR,
    "robust_int8_clean.json"
)

ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ============================================================
# 2. SAFETY CHECK
# ============================================================

assert os.path.exists(
    FP32_CHECKPOINT
)

assert os.path.exists(
    TRAIN_DIR
)

assert os.path.exists(
    TEST_DIR
)

assert os.path.abspath(
    RESULTS_DIR
) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
)

print("Safety check passed.")
print("Original results will NOT be modified.")


# ============================================================
# 3. FORCE CPU FOR INT8
# ============================================================

device = torch.device("cpu")

torch.backends.quantized.engine = "fbgemm"

print("\nINT8 device:", device)
print("Quantization backend:",
      torch.backends.quantized.engine)


# ============================================================
# 4. TRANSFORM
# ============================================================

eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


# ============================================================
# 5. DATASETS
# ============================================================

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=eval_transform
)

print("\nTraining images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", train_dataset.class_to_idx)


# ============================================================
# 6. REBUILD ROBUST FP32 MODEL
# ============================================================

model = shufflenet_v2_x1_0(
    weights=None
)

model.fc = nn.Linear(
    model.fc.in_features,
    2
)

state_dict = torch.load(
    FP32_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

missing_keys, unexpected_keys = (
    model.load_state_dict(
        state_dict,
        strict=True
    )
)

assert len(missing_keys) == 0
assert len(unexpected_keys) == 0

model.eval()

print("\nRobust FP32 model loaded successfully.")


# ============================================================
# 7. FX QUANTIZATION CONFIGURATION
# ============================================================

qconfig = get_default_qconfig(
    "fbgemm"
)

qconfig_mapping = {
    "": qconfig
}

print("\nFX quantization configuration ready.")


# ============================================================
# 8. PREPARE FX GRAPH
# ============================================================

example_inputs = (
    torch.randn(
        1,
        3,
        224,
        224
    ),
)

print("\nPreparing FX graph...")

prepared_model = prepare_fx(
    model,
    qconfig_mapping,
    example_inputs
)

print("FX graph prepared successfully.")


# ============================================================
# 9. CALIBRATION
# ============================================================

calibration_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

num_calibration_batches = 10

print(
    f"\nCalibrating with "
    f"{num_calibration_batches} batches..."
)

with torch.no_grad():

    for batch_index, (images, _) in enumerate(
        calibration_loader
    ):

        prepared_model(
            images
        )

        if (
            batch_index + 1
            >= num_calibration_batches
        ):
            break

print("Calibration complete.")


# ============================================================
# 10. CONVERT TO INT8
# ============================================================

print("\nConverting FX graph to INT8...")

model_int8 = convert_fx(
    prepared_model
)

model_int8.eval()

print("FX INT8 conversion successful.")


# ============================================================
# 11. TEST INT8 FORWARD PASS
# ============================================================

print("\nTesting INT8 forward pass...")

with torch.no_grad():

    test_input = torch.randn(
        1,
        3,
        224,
        224
    )

    test_output = model_int8(
        test_input
    )

print(
    "Forward pass successful."
)

print(
    "Output shape:",
    tuple(test_output.shape)
)


# ============================================================
# 12. SAVE INT8 MODEL
# ============================================================

torch.save(
    model_int8.state_dict(),
    INT8_CHECKPOINT
)

print("\nINT8 checkpoint saved:")
print(INT8_CHECKPOINT)


# ============================================================
# 13. EVALUATE CLEAN TEST SET
# ============================================================

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)

all_labels = []
all_predictions = []

print(
    "\nEvaluating INT8 model "
    "on clean test set..."
)

with torch.no_grad():

    for images, labels in test_loader:

        outputs = model_int8(
            images
        )

        predictions = torch.argmax(
            outputs,
            dim=1
        )

        all_labels.extend(
            labels.numpy().tolist()
        )

        all_predictions.extend(
            predictions.numpy().tolist()
        )


# ============================================================
# 14. METRICS
# ============================================================

accuracy = accuracy_score(
    all_labels,
    all_predictions
)

precision = precision_score(
    all_labels,
    all_predictions,
    zero_division=0
)

recall = recall_score(
    all_labels,
    all_predictions,
    zero_division=0
)

f1 = f1_score(
    all_labels,
    all_predictions,
    zero_division=0
)


# ============================================================
# 15. MODEL SIZE
# ============================================================

fp32_size_mb = (
    os.path.getsize(
        FP32_CHECKPOINT
    ) / (1024 ** 2)
)

int8_size_mb = (
    os.path.getsize(
        INT8_CHECKPOINT
    ) / (1024 ** 2)
)

size_reduction = (
    (fp32_size_mb - int8_size_mb)
    / fp32_size_mb
    * 100
)


# ============================================================
# 16. SAVE RESULTS
# ============================================================

results = {

    "model":
        "ShuffleNetV2",

    "quantization":
        "FX Graph-Mode Static INT8",

    "backend":
        "fbgemm",

    "dataset":
        "AI-vs-Real",

    "evaluation":
        "clean_test",

    "accuracy":
        float(accuracy),

    "precision":
        float(precision),

    "recall":
        float(recall),

    "f1":
        float(f1),

    "fp32_checkpoint":
        FP32_CHECKPOINT,

    "int8_checkpoint":
        INT8_CHECKPOINT,

    "fp32_size_mb":
        float(fp32_size_mb),

    "int8_size_mb":
        float(int8_size_mb),

    "size_reduction_percent":
        float(size_reduction),

    "calibration_batches":
        num_calibration_batches,

    "input_size":
        "224x224"
}

with open(
    OUTPUT_FILE,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


# ============================================================
# 17. FINAL OUTPUT
# ============================================================

print("\n" + "=" * 60)
print("ROBUST INT8 QUANTIZATION COMPLETE")
print("=" * 60)

print(
    f"\nAccuracy : {accuracy:.4f}"
)

print(
    f"Precision: {precision:.4f}"
)

print(
    f"Recall   : {recall:.4f}"
)

print(
    f"F1 Score : {f1:.4f}"
)

print(
    f"\nFP32 size: "
    f"{fp32_size_mb:.2f} MB"
)

print(
    f"INT8 size: "
    f"{int8_size_mb:.2f} MB"
)

print(
    f"Reduction: "
    f"{size_reduction:.2f}%"
)

print("\nINT8 checkpoint:")
print(INT8_CHECKPOINT)

print("\nResults:")
print(OUTPUT_FILE)

print("\nOriginal results directory was not modified.")

Safety check passed.
Original results will NOT be modified.

INT8 device: cpu
Quantization backend: fbgemm

Training images: 3500
Test images: 750
Classes: {'ai': 0, 'nature': 1}

Robust FP32 model loaded successfully.

FX quantization configuration ready.

Preparing FX graph...


/tmp/ipykernel_841/1064346637.py:223: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model = prepare_fx(
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(

FX graph prepared successfully.

Calibrating with 10 batches...
Calibration complete.

Converting FX graph to INT8...


/tmp/ipykernel_841/1064346637.py:275: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = convert_fx(


FX INT8 conversion successful.

Testing INT8 forward pass...
Forward pass successful.
Output shape: (1, 2)

INT8 checkpoint saved:
/content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints/shufflenetv2_robust_int8.pth

Evaluating INT8 model on clean test set...

ROBUST INT8 QUANTIZATION COMPLETE

Accuracy : 0.9200
Precision: 0.9758
Recall   : 0.8613
F1 Score : 0.9150

FP32 size: 4.96 MB
INT8 size: 1.45 MB
Reduction: 70.73%

INT8 checkpoint:
/content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints/shufflenetv2_robust_int8.pth

Results:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results/robust_int8_clean.json

Original results directory was not modified.


In [21]:
# ============================================================
# ROBUST SHUFFLENETV2 INT8 CORRUPTION EVALUATION
# ============================================================

import os
import io
import json
import numpy as np

import torch
from PIL import Image, ImageFilter

from torchvision import datasets, transforms
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. PATHS
# ============================================================

PROJECT_DIR = "/content/drive/MyDrive/diffusion_project"

ROBUSTNESS_DIR = os.path.join(
    PROJECT_DIR,
    "robustness_experiment"
)

INT8_CHECKPOINT = os.path.join(
    ROBUSTNESS_DIR,
    "checkpoints",
    "shufflenetv2_robust_int8.pth"
)

TEST_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "test"
)

RESULTS_DIR = os.path.join(
    ROBUSTNESS_DIR,
    "results"
)

OUTPUT_FILE = os.path.join(
    RESULTS_DIR,
    "robust_int8_corruptions.json"
)

ORIGINAL_RESULTS_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "shufflenetv2"
)

os.makedirs(
    RESULTS_DIR,
    exist_ok=True
)


# ============================================================
# 2. SAFETY CHECKS
# ============================================================

assert os.path.exists(
    INT8_CHECKPOINT
), (
    f"INT8 checkpoint not found:\n"
    f"{INT8_CHECKPOINT}"
)

assert os.path.exists(
    TEST_DIR
), (
    f"Test dataset not found:\n"
    f"{TEST_DIR}"
)

assert os.path.abspath(
    RESULTS_DIR
) != os.path.abspath(
    ORIGINAL_RESULTS_DIR
), (
    "Safety error: original results directory "
    "cannot be used."
)

print("INT8 checkpoint:")
print(INT8_CHECKPOINT)

print("\nResults directory:")
print(RESULTS_DIR)

print("\nSafety check passed.")
print("Original results will NOT be modified.")


# ============================================================
# 3. FORCE CPU
# ============================================================

device = torch.device("cpu")

torch.backends.quantized.engine = "fbgemm"

print("\nDevice:", device)
print(
    "Quantization backend:",
    torch.backends.quantized.engine
)


# ============================================================
# 4. CORRUPTION CLASSES
# ============================================================

class JPEGCompression:

    def __init__(self, quality=30):
        self.quality = quality

    def __call__(self, image):

        buffer = io.BytesIO()

        image.save(
            buffer,
            format="JPEG",
            quality=self.quality
        )

        buffer.seek(0)

        return Image.open(
            buffer
        ).convert("RGB")


class GaussianNoise:

    def __init__(self, std=0.08):
        self.std = std

    def __call__(self, image):

        image_np = (
            np.asarray(image)
            .astype(np.float32)
            / 255.0
        )

        noise = np.random.normal(
            0.0,
            self.std,
            image_np.shape
        )

        noisy = image_np + noise

        noisy = np.clip(
            noisy,
            0.0,
            1.0
        )

        noisy = (
            noisy * 255.0
        ).astype(
            np.uint8
        )

        return Image.fromarray(
            noisy
        )


class FixedGaussianBlur:

    def __init__(self, radius=2.0):
        self.radius = radius

    def __call__(self, image):

        return image.filter(
            ImageFilter.GaussianBlur(
                radius=self.radius
            )
        )


class LowResolution:

    def __init__(self, low_size=56):
        self.low_size = low_size

    def __call__(self, image):

        image = image.resize(
            (
                self.low_size,
                self.low_size
            ),
            Image.Resampling.BILINEAR
        )

        image = image.resize(
            (224, 224),
            Image.Resampling.BILINEAR
        )

        return image


# ============================================================
# 5. NORMALIZATION
# ============================================================

NORMALIZE = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)


# ============================================================
# 6. TRANSFORM BUILDER
# ============================================================

def make_transform(corruption=None):

    transform_list = []

    if corruption is not None:

        transform_list.append(
            corruption
        )

    transform_list.extend([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        NORMALIZE
    ])

    return transforms.Compose(
        transform_list
    )


# ============================================================
# 7. REBUILD THE SAME INT8 ARCHITECTURE
# ============================================================

from torchvision.models import shufflenet_v2_x1_0
import torch.nn as nn

fp32_model = shufflenet_v2_x1_0(
    weights=None
)

fp32_model.fc = nn.Linear(
    fp32_model.fc.in_features,
    2
)

fp32_model.eval()


# ============================================================
# 8. PREPARE FX QUANTIZATION STRUCTURE
# ============================================================

from torch.ao.quantization import (
    get_default_qconfig
)

from torch.ao.quantization.quantize_fx import (
    prepare_fx,
    convert_fx
)

qconfig = get_default_qconfig(
    "fbgemm"
)

qconfig_mapping = {
    "": qconfig
}

example_inputs = (
    torch.randn(
        1,
        3,
        224,
        224
    ),
)

prepared_model = prepare_fx(
    fp32_model,
    qconfig_mapping,
    example_inputs
)


# ============================================================
# 9. LOAD TRAINING DATA FOR CALIBRATION
# ============================================================

TRAIN_DIR = os.path.join(
    PROJECT_DIR,
    "dataset_final",
    "train"
)

calibration_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    NORMALIZE
])

calibration_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=calibration_transform
)

calibration_loader = DataLoader(
    calibration_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2
)


# ============================================================
# 10. CALIBRATION
# ============================================================

print("\nCalibrating INT8 model...")

num_calibration_batches = 10

with torch.no_grad():

    for batch_index, (images, _) in enumerate(
        calibration_loader
    ):

        prepared_model(
            images
        )

        if (
            batch_index + 1
            >= num_calibration_batches
        ):
            break

print(
    f"Calibration complete using "
    f"{num_calibration_batches} batches."
)


# ============================================================
# 11. CONVERT TO INT8
# ============================================================

model_int8 = convert_fx(
    prepared_model
)

model_int8.eval()

print(
    "INT8 model reconstructed successfully."
)


# ============================================================
# 12. LOAD SAVED INT8 CHECKPOINT
# ============================================================

saved_state = torch.load(
    INT8_CHECKPOINT,
    map_location="cpu",
    weights_only=True
)

missing_keys, unexpected_keys = (
    model_int8.load_state_dict(
        saved_state,
        strict=True
    )
)

assert len(missing_keys) == 0
assert len(unexpected_keys) == 0

print(
    "Saved INT8 checkpoint loaded successfully."
)


# ============================================================
# 13. EVALUATION FUNCTION
# ============================================================

def evaluate_dataset(dataset):

    loader = DataLoader(
        dataset,
        batch_size=32,
        shuffle=False,
        num_workers=2
    )

    all_labels = []
    all_predictions = []

    with torch.no_grad():

        for images, labels in loader:

            outputs = model_int8(
                images
            )

            predictions = torch.argmax(
                outputs,
                dim=1
            )

            all_labels.extend(
                labels.numpy().tolist()
            )

            all_predictions.extend(
                predictions.numpy().tolist()
            )

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    precision = precision_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    recall = recall_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    f1 = f1_score(
        all_labels,
        all_predictions,
        zero_division=0
    )

    return {
        "accuracy": float(accuracy),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "num_images": len(dataset)
    }


# ============================================================
# 14. SAME FIVE CONDITIONS
# ============================================================

corruptions = {

    "clean": None,

    "jpeg": JPEGCompression(
        quality=30
    ),

    "gaussian_blur": FixedGaussianBlur(
        radius=2.0
    ),

    "gaussian_noise": GaussianNoise(
        std=0.08
    ),

    "low_resolution": LowResolution(
        low_size=56
    )
}


# ============================================================
# 15. RUN EVALUATION
# ============================================================

results = {

    "model":
        "ShuffleNetV2",

    "quantization":
        "FX Graph-Mode Static INT8",

    "backend":
        "fbgemm",

    "dataset":
        "AI-vs-Real",

    "evaluation_type":
        "corruption_test",

    "corruptions": {}
}


print("\n" + "=" * 60)
print("ROBUST INT8 CORRUPTION EVALUATION")
print("=" * 60)


for corruption_name, corruption in corruptions.items():

    print(
        f"\nEvaluating: {corruption_name}"
    )

    dataset = datasets.ImageFolder(
        TEST_DIR,
        transform=make_transform(
            corruption
        )
    )

    metrics = evaluate_dataset(
        dataset
    )

    results[
        "corruptions"
    ][
        corruption_name
    ] = metrics

    print(
        f"Accuracy : "
        f"{metrics['accuracy']:.4f}"
    )

    print(
        f"Precision: "
        f"{metrics['precision']:.4f}"
    )

    print(
        f"Recall   : "
        f"{metrics['recall']:.4f}"
    )

    print(
        f"F1 Score : "
        f"{metrics['f1']:.4f}"
    )


# ============================================================
# 16. SAVE RESULTS
# ============================================================

with open(
    OUTPUT_FILE,
    "w"
) as f:

    json.dump(
        results,
        f,
        indent=4
    )


# ============================================================
# 17. COMPLETE
# ============================================================

print("\n" + "=" * 60)
print("ROBUST INT8 CORRUPTION EVALUATION COMPLETE")
print("=" * 60)

print("\nSaved results:")
print(OUTPUT_FILE)

print("\nOriginal results directory was not modified.")

INT8 checkpoint:
/content/drive/MyDrive/diffusion_project/robustness_experiment/checkpoints/shufflenetv2_robust_int8.pth

Results directory:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results

Safety check passed.
Original results will NOT be modified.

Device: cpu
Quantization backend: fbgemm


/tmp/ipykernel_841/1706843759.py:306: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  prepared_model = prepare_fx(
/usr/local/lib/python3.13/dist-packages/torch/ao/quantization/quantize_fx.py:146: FutureWarning: Passing a QConfig dictionary to prepare is deprecated and will not be supported in a future version. Please pass in a QConfigMapping instead.
  prepared = prepare(


Calibrating INT8 model...
Calibration complete using 10 batches.


/tmp/ipykernel_841/1706843759.py:376: DeprecationWarning: torch.ao.quantization is deprecated and will be removed in 2.10. 
For migrations of users: 
1. Eager mode quantization (torch.ao.quantization.quantize, torch.ao.quantization.quantize_dynamic), please migrate to use torchao eager mode quantize_ API instead 
2. FX graph mode quantization (torch.ao.quantization.quantize_fx.prepare_fx,torch.ao.quantization.quantize_fx.convert_fx, please migrate to use torchao pt2e quantization API instead (prepare_pt2e, convert_pt2e) 
3. pt2e quantization has been migrated to torchao (https://github.com/pytorch/ao/tree/main/torchao/quantization/pt2e) 
see https://github.com/pytorch/ao/issues/2259 for more details
  model_int8 = convert_fx(


INT8 model reconstructed successfully.
Saved INT8 checkpoint loaded successfully.

ROBUST INT8 CORRUPTION EVALUATION

Evaluating: clean


/usr/local/lib/python3.13/dist-packages/torch/_utils.py:476: UserWarning: TypedStorage is deprecated. It will be removed in the future and UntypedStorage will be the only storage class. This should only matter to you if you are using storages directly.  To access UntypedStorage directly, use tensor.untyped_storage() instead of tensor.storage()
  device=storage.device,


Accuracy : 0.9200
Precision: 0.9758
Recall   : 0.8613
F1 Score : 0.9150

Evaluating: jpeg
Accuracy : 0.8680
Precision: 0.8317
Recall   : 0.9227
F1 Score : 0.8748

Evaluating: gaussian_blur
Accuracy : 0.8907
Precision: 0.9246
Recall   : 0.8507
F1 Score : 0.8861

Evaluating: gaussian_noise
Accuracy : 0.8627
Precision: 0.8842
Recall   : 0.8347
F1 Score : 0.8587

Evaluating: low_resolution
Accuracy : 0.8720
Precision: 0.8479
Recall   : 0.9067
F1 Score : 0.8763

ROBUST INT8 CORRUPTION EVALUATION COMPLETE

Saved results:
/content/drive/MyDrive/diffusion_project/robustness_experiment/results/robust_int8_corruptions.json

Original results directory was not modified.
